# Stress Prediction V32 — Validated Team Blend

V14 OOF를 재생성하고 동일한 KFold의 팀 최고 V7 OOF와 80:20으로 혼합하여 Fold별 MAE와 평균 MAE를 출력합니다.


In [2]:
from pathlib import Path
import os

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

## 1. 설정

In [3]:
DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project")

TARGET = "stress_score"
ID_COLUMN = "ID"
SEEDS = (42, 77, 2026)
N_ESTIMATORS = 1200
BASE_QUANTILE = 0.510
POOLED_QUANTILE = 0.505
AGE_QUANTILE = 0.500

# 기준 모델을 크게 유지하는 보수적 탐색
BASE_WEIGHTS = (0.80, 0.85, 0.90, 0.95, 1.00)
ROUND_OPTIONS = (False, True)

WEIGHTED_FEATURES = [
    "mean_working", "bmi", "cholesterol", "height",
    "glucose", "weight", "cholesterol_glucose_ratio",
    "bone_density",
]
AGE_WEIGHTED_FEATURES = WEIGHTED_FEATURES + [
    "mean_arterial_pressure", "diastolic_blood_pressure",
]

## 2. 데이터 불러오기 및 검증

In [4]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

if set(train.columns) - {TARGET} != set(test.columns):
    raise ValueError("Train과 Test 입력 컬럼이 다릅니다.")
if list(submission.columns) != [ID_COLUMN, TARGET]:
    raise ValueError("제출 양식 컬럼이 올바르지 않습니다.")
if not submission[ID_COLUMN].equals(test[ID_COLUMN]):
    raise ValueError("제출 양식과 Test의 ID 순서가 다릅니다.")
if train[TARGET].isna().any():
    raise ValueError("Train 타깃에 결측값이 있습니다.")

y = train[TARGET].reset_index(drop=True)
raw_train = train.drop(columns=[TARGET, ID_COLUMN]).reset_index(drop=True)
raw_test = test.drop(columns=[ID_COLUMN]).reset_index(drop=True)
print("Train:", train.shape, "| Test:", test.shape)

Train: (3000, 18) | Test: (3000, 17)


## 3. 행 단위 파생변수

In [5]:
def add_base_features(frame):
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = (
        result["cholesterol"] / (result["glucose"] + 1e-6)
    )
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_weighted_features(frame):
    result = add_base_features(frame)
    for feature in WEIGHTED_FEATURES:
        result[f"{feature}__copy1"] = result[feature]
        result[f"{feature}__copy2"] = result[feature]
    return result


def add_age_features(frame):
    result = add_base_features(frame)
    result["age_bmi"] = result["age"] * result["bmi"]
    result["age_glucose"] = result["age"] * result["glucose"]
    result["age_cholesterol"] = result["age"] * result["cholesterol"]
    result["age_bone_density"] = result["age"] * result["bone_density"]
    result["age_mean_arterial_pressure"] = (
        result["age"] * result["mean_arterial_pressure"]
    )
    for feature in AGE_WEIGHTED_FEATURES:
        result[f"{feature}__copy1"] = result[feature]
    return result


X_weighted = add_weighted_features(raw_train)
X_test_weighted = add_weighted_features(raw_test)
X_age = add_age_features(raw_train)
X_test_age = add_age_features(raw_test)
print("Weighted:", X_weighted.shape, "| Age:", X_age.shape)

Weighted: (3000, 38) | Age: (3000, 37)


## 4. Train 전용 전처리

In [6]:
def make_preprocessor(frame):
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer(
        [
            (
                "categorical",
                Pipeline(
                    [
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="constant",
                                fill_value="missing",
                            ),
                        ),
                        (
                            "encoder",
                            OneHotEncoder(
                                handle_unknown="ignore",
                                sparse_output=False,
                            ),
                        ),
                    ]
                ),
                categorical,
            ),
            (
                "numerical",
                SimpleImputer(strategy="median", add_indicator=True),
                numerical,
            ),
        ]
    )


def tree_predictions(model, matrix):
    return np.column_stack(
        [tree.predict(matrix) for tree in model.estimators_]
    )


def clipped_quantile(matrix, quantile):
    return np.clip(
        np.quantile(matrix, quantile, axis=1),
        0.0,
        1.0,
    )

## 5. 누수 없는 5-Fold OOF 예측

In [7]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
fold_indices = []
oof_base = np.zeros(len(y))
oof_pooled = np.zeros(len(y))
oof_age = np.zeros(len(y))

for fold, (train_idx, valid_idx) in enumerate(cv.split(X_weighted), start=1):
    fold_indices.append(valid_idx)

    weighted_preprocessor = make_preprocessor(
        X_weighted.iloc[train_idx]
    )
    weighted_train = weighted_preprocessor.fit_transform(
        X_weighted.iloc[train_idx]
    )
    weighted_valid = weighted_preprocessor.transform(
        X_weighted.iloc[valid_idx]
    )

    pooled_trees = []
    seed42_trees = None
    for seed in SEEDS:
        model = ExtraTreesRegressor(
            n_estimators=N_ESTIMATORS,
            min_samples_leaf=1,
            max_features=1,
            random_state=seed,
            n_jobs=-1,
        )
        model.fit(weighted_train, y.iloc[train_idx])
        prediction_matrix = tree_predictions(model, weighted_valid)
        pooled_trees.append(prediction_matrix)
        if seed == 42:
            seed42_trees = prediction_matrix

    oof_base[valid_idx] = clipped_quantile(
        seed42_trees,
        BASE_QUANTILE,
    )
    oof_pooled[valid_idx] = clipped_quantile(
        np.concatenate(pooled_trees, axis=1),
        POOLED_QUANTILE,
    )

    age_preprocessor = make_preprocessor(X_age.iloc[train_idx])
    age_train = age_preprocessor.fit_transform(X_age.iloc[train_idx])
    age_valid = age_preprocessor.transform(X_age.iloc[valid_idx])
    age_model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS,
        min_samples_leaf=1,
        max_features=1,
        random_state=42,
        n_jobs=-1,
    )
    age_model.fit(age_train, y.iloc[train_idx])
    oof_age[valid_idx] = clipped_quantile(
        tree_predictions(age_model, age_valid),
        AGE_QUANTILE,
    )

    base_score = mean_absolute_error(
        y.iloc[valid_idx],
        oof_base[valid_idx],
    )
    print(f"Fold {fold} base MAE: {base_score:.6f}")

Fold 1 base MAE: 0.136627
Fold 2 base MAE: 0.150506
Fold 3 base MAE: 0.149009
Fold 4 base MAE: 0.159168
Fold 5 base MAE: 0.150243


## 6. 보수적 블렌딩 탐색

In [8]:
# V10 계열 보조 예측
oof_auxiliary = np.clip(
    0.70 * oof_pooled + 0.30 * oof_age,
    0.0,
    1.0,
)

rows = []
candidate_predictions = {}

for base_weight in BASE_WEIGHTS:
    for use_rounding in ROUND_OPTIONS:
        prediction = (
            base_weight * oof_base
            + (1.0 - base_weight) * oof_auxiliary
        )
        if use_rounding:
            prediction = np.round(prediction, 2)
        prediction = np.clip(prediction, 0.0, 1.0)
        key = (base_weight, use_rounding)
        candidate_predictions[key] = prediction

        fold_scores = [
            mean_absolute_error(y.iloc[idx], prediction[idx])
            for idx in fold_indices
        ]
        overall_mae = mean_absolute_error(y, prediction)
        fold_std = float(np.std(fold_scores))
        # 미세한 CV 우연을 줄이기 위한 안정성 점수
        robust_score = overall_mae + 0.15 * fold_std
        rows.append(
            {
                "base_weight": base_weight,
                "auxiliary_weight": 1.0 - base_weight,
                "round_2_decimals": use_rounding,
                "CV_MAE": overall_mae,
                "fold_std": fold_std,
                "robust_score": robust_score,
                "fold_wins_vs_base": 0,
            }
        )

base_fold_scores = [
    mean_absolute_error(y.iloc[idx], oof_base[idx])
    for idx in fold_indices
]
for row in rows:
    pred = candidate_predictions[
        (row["base_weight"], row["round_2_decimals"])
    ]
    scores = [
        mean_absolute_error(y.iloc[idx], pred[idx])
        for idx in fold_indices
    ]
    row["fold_wins_vs_base"] = sum(
        score < base_score
        for score, base_score in zip(scores, base_fold_scores)
    )

results = pd.DataFrame(rows).sort_values(
    ["robust_score", "CV_MAE"]
).reset_index(drop=True)

# 최소 3개 Fold에서 기준 모델을 이긴 후보만 허용합니다.
eligible = results[results["fold_wins_vs_base"] >= 3]
if len(eligible):
    selected = eligible.iloc[0]
else:
    selected = results[
        (results["base_weight"] == 1.0)
        & (~results["round_2_decimals"])
    ].iloc[0]

print("Base CV MAE:", f"{mean_absolute_error(y, oof_base):.6f}")
print("Selected CV MAE:", f"{selected['CV_MAE']:.6f}")
print("Selected base weight:", selected["base_weight"])
print("Selected auxiliary weight:", selected["auxiliary_weight"])
print("Round to 2 decimals:", selected["round_2_decimals"])
print("Fold wins vs base:", int(selected["fold_wins_vs_base"]))
results

Base CV MAE: 0.149110
Selected CV MAE: 0.148953
Selected base weight: 0.8
Selected auxiliary weight: 0.19999999999999996
Round to 2 decimals: True
Fold wins vs base: 5


,base_weight,auxiliary_weight,round_2_decimals,CV_MAE,fold_std,robust_score,fold_wins_vs_base
0,0.80,0.20,True,0.148953,0.007221,0.150037,5
1,0.80,0.20,False,0.148988,0.007257,0.150076,4
2,0.85,0.15,False,0.149019,0.007246,0.150105,4
3,0.90,0.10,False,0.149049,0.007236,0.150134,4
4,0.90,0.10,True,0.149060,0.007235,0.150145,4
5,0.85,0.15,True,0.149070,0.007238,0.150156,3
6,0.95,0.05,False,0.149080,0.007225,0.150164,4
7,0.95,0.05,True,0.149097,0.007253,0.150185,3
8,1.00,0.00,False,0.149110,0.007214,0.150193,0
9,1.00,0.00,True,0.149137,0.007208,0.150218,1


## 7. 전체 Train 학습 및 Test 예측

In [9]:
weighted_preprocessor = make_preprocessor(X_weighted)
weighted_train = weighted_preprocessor.fit_transform(X_weighted)
weighted_test = weighted_preprocessor.transform(X_test_weighted)

pooled_test_trees = []
seed42_test_trees = None
for seed in SEEDS:
    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS,
        min_samples_leaf=1,
        max_features=1,
        random_state=seed,
        n_jobs=-1,
    )
    model.fit(weighted_train, y)
    matrix = tree_predictions(model, weighted_test)
    pooled_test_trees.append(matrix)
    if seed == 42:
        seed42_test_trees = matrix
    print(f"Weighted model seed {seed}: complete")

pred_base = clipped_quantile(seed42_test_trees, BASE_QUANTILE)
pred_pooled = clipped_quantile(
    np.concatenate(pooled_test_trees, axis=1),
    POOLED_QUANTILE,
)

age_preprocessor = make_preprocessor(X_age)
age_train = age_preprocessor.fit_transform(X_age)
age_test = age_preprocessor.transform(X_test_age)
age_model = ExtraTreesRegressor(
    n_estimators=N_ESTIMATORS,
    min_samples_leaf=1,
    max_features=1,
    random_state=42,
    n_jobs=-1,
)
age_model.fit(age_train, y)
pred_age = clipped_quantile(
    tree_predictions(age_model, age_test),
    AGE_QUANTILE,
)
pred_auxiliary = np.clip(
    0.70 * pred_pooled + 0.30 * pred_age,
    0.0,
    1.0,
)
print("Final models: complete")

Weighted model seed 42: complete
Weighted model seed 77: complete
Weighted model seed 2026: complete
Final models: complete


## 8. V14 제출 후보 생성

In [10]:
def make_blend(base_weight, use_rounding):
    prediction = (
        base_weight * pred_base
        + (1.0 - base_weight) * pred_auxiliary
    )
    if use_rounding:
        prediction = np.round(prediction, 2)
    return np.clip(prediction, 0.0, 1.0)


selected_base_weight = float(selected["base_weight"])
selected_rounding = bool(selected["round_2_decimals"])
pred_v14 = make_blend(selected_base_weight, selected_rounding)

# 제출 선택권을 남기기 위한 고정 보수 후보
pred_v14_blend95 = make_blend(0.95, False)
pred_v14_blend90 = make_blend(0.90, False)

outputs = {
    "submit_v14_main.csv": pred_v14,
    "submit_v14_blend95.csv": pred_v14_blend95,
    "submit_v14_blend90.csv": pred_v14_blend90,
}
for filename, prediction in outputs.items():
    frame = submission.copy()
    frame[ID_COLUMN] = test[ID_COLUMN]
    frame[TARGET] = prediction
    path = OUTPUT_DIR / filename
    frame.to_csv(path, index=False, encoding="utf-8")
    print("Saved:", path)

print("\nV14 main prediction range:", pred_v14.min(), pred_v14.max())
print("첫 10개 예측값:", pred_v14[:10])
submission_v14 = submission.copy()
submission_v14[TARGET] = pred_v14
submission_v14.head(10)

Saved: /Users/bitsaem/Desktop/stress_project/submit_v14_main.csv
Saved: /Users/bitsaem/Desktop/stress_project/submit_v14_blend95.csv
Saved: /Users/bitsaem/Desktop/stress_project/submit_v14_blend90.csv

V14 main prediction range: 0.0 1.0
첫 10개 예측값: [0.56 0.97 0.19 0.44 0.53 0.46 0.78 0.47 0.55 0.88]


,ID,stress_score
0,TEST_0000,0.56
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.44
4,TEST_0004,0.53
5,TEST_0005,0.46
6,TEST_0006,0.78
7,TEST_0007,0.47
8,TEST_0008,0.55
9,TEST_0009,0.88


## 제출 권장 순서

1. `submit_v14_main.csv` — 반복 검증의 안정성 기준으로 자동 선택된 주 제출본
2. 제출 여유가 있을 때만 `submit_v14_blend95.csv`
3. `submit_v14_blend90.csv`는 다양성 확인용이며 우선순위가 가장 낮음

기존 최고 기록 제출 파일은 삭제하지 말고 반드시 보관하세요.

In [11]:
# V14와 팀 최고 V7을 동일한 KFold(random_state=42) OOF 기준으로 비교합니다.
team_oof_path = OUTPUT_DIR / "oof_team_best_v7_verified.csv"
team_oof = pd.read_csv(team_oof_path)

if not team_oof[ID_COLUMN].equals(train[ID_COLUMN]):
    raise ValueError("팀 최고 OOF와 Train의 ID 순서가 다릅니다.")

oof_team_best = team_oof["oof_seed_42"].to_numpy(float)
oof_v14_blend95 = np.clip(
    0.95 * oof_base + 0.05 * oof_auxiliary,
    0.0,
    1.0,
)
oof_v32 = np.clip(
    np.round(0.80 * oof_team_best + 0.20 * oof_v14_blend95, 2),
    0.0,
    1.0,
)

fold_rows = []
for fold, valid_idx in enumerate(fold_indices, 1):
    team_score = mean_absolute_error(y.iloc[valid_idx], oof_team_best[valid_idx])
    v14_score = mean_absolute_error(y.iloc[valid_idx], oof_v14_blend95[valid_idx])
    v32_score = mean_absolute_error(y.iloc[valid_idx], oof_v32[valid_idx])
    fold_rows.append({
        "Fold": fold,
        "Team_best_MAE": team_score,
        "V14_blend95_MAE": v14_score,
        "V32_MAE": v32_score,
        "V32_vs_team_best": v32_score - team_score,
    })

fold_result = pd.DataFrame(fold_rows)
display(fold_result)
print("Team best CV MAE:", f"{mean_absolute_error(y, oof_team_best):.6f}")
print("V14 blend95 CV MAE:", f"{mean_absolute_error(y, oof_v14_blend95):.6f}")
print("V32 CV MAE:", f"{mean_absolute_error(y, oof_v32):.6f}")
print(
    "V32 vs team best:",
    f"{mean_absolute_error(y, oof_v32) - mean_absolute_error(y, oof_team_best):+.6f}",
)

# 동일한 80:20 비율로 Test 제출값을 생성합니다.
team_best_submission = pd.read_csv(
    OUTPUT_DIR / "submit_team_best_original_pair_neighbor.csv"
)
v14_submission = pd.read_csv(OUTPUT_DIR / "submit_v14_blend95.csv")
if not team_best_submission[ID_COLUMN].equals(v14_submission[ID_COLUMN]):
    raise ValueError("두 제출 파일의 ID 순서가 다릅니다.")

test_prediction = np.clip(
    np.round(
        0.80 * team_best_submission[TARGET].to_numpy(float)
        + 0.20 * v14_submission[TARGET].to_numpy(float),
        2,
    ),
    0.0,
    1.0,
)
submission_v32 = team_best_submission.copy()
submission_v32[TARGET] = test_prediction
v32_path = OUTPUT_DIR / "submit_v32_team_best80_v14_20.csv"
submission_v32.to_csv(v32_path, index=False, encoding="utf-8")
print("Saved:", v32_path)
display(submission_v32.head(10))



,Fold,Team_best_MAE,V14_blend95_MAE,V32_MAE,V32_vs_team_best
0,1,0.134500,0.136566,0.134817,0.000317
1,2,0.150167,0.150534,0.150200,0.000033
2,3,0.148250,0.148960,0.148250,0.000000
3,4,0.159217,0.159131,0.159100,-0.000117
4,5,0.150617,0.150208,0.150517,-0.000100


Team best CV MAE: 0.148550
V14 blend95 CV MAE: 0.149080
V32 CV MAE: 0.148577
V32 vs team best: +0.000027
Saved: /Users/bitsaem/Desktop/stress_project/submit_v32_team_best80_v14_20.csv


,ID,stress_score
0,TEST_0000,0.56
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.47
4,TEST_0004,0.53
5,TEST_0005,0.47
6,TEST_0006,0.78
7,TEST_0007,0.48
8,TEST_0008,0.57
9,TEST_0009,0.88
